# Goofy domain-adaptation test (PoseAnything cartoon CAPE)Runs `test_tta.py` on the 39 labeled Goofy frames: 5 support / 34 query, 3 trials.One run prints **both** the no-TTA and with-TTA columns.**Kaggle dataset** must contain:```PoseAnything_ours/        <- the repo folder (4 MB)goofy_eval.zip            <- 39 images + coco_goofy.jsonckpt/*.pth                <- slim checkpoints```Accelerator: **GPU T4 x1** (P100 also fine).

## 0. Version gate — run this FIRST

In [ ]:
import sys, platformprint("python", platform.python_version())assert sys.version_info[:2] in [(3,8),(3,9),(3,10),(3,11)], (    "mmcv-full 1.x has no wheel for this Python. STOP: use the server instead.")print("OK: a prebuilt mmcv-full 1.7.2 wheel exists for this Python")

## 1. Environment (~6 min, then **RESTART THE KERNEL**)mmcv-full 1.x must match the torch it was built against, so torch is pinned to2.0.1+cu118 and mmcv-full comes from the matching OpenMMLab wheel index.mmpose 0.29.0 caps mmcv at 1.7.0 but only 1.7.2 is published for torch 2.0 —1.7.2 is API-compatible, so the cap is raised by one line.

In [ ]:
!pip -q uninstall -y torch torchvision torchaudio fastai 2>/dev/null!pip -q install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118!pip -q install "numpy<2" cython!pip -q install mmcv-full==1.7.2 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html!pip -q install mmpose==0.29.0 xtcocotools json_tricks timm torchmetrics munkres "yapf==0.40.1"

In [ ]:
# raise mmpose's mmcv upper bound WITHOUT importing mmpose (the import is what asserts)import site, pathlibdone = Falsefor sp in site.getsitepackages() + [site.getusersitepackages()]:    f = pathlib.Path(sp) / "mmpose" / "__init__.py"    if f.exists():        s = f.read_text()        s = s.replace("mmcv_maximum_version = '1.7.0'", "mmcv_maximum_version = '1.7.3'")        f.write_text(s); print("patched", f); done = Trueassert done, "mmpose not found - did the pip install above fail?"

### >>> Now do `Run` -> `Restart kernel`, then continue from cell 2. <<<

## 2. Sanity check the environment

In [ ]:
import torch, mmcv, mmposeprint("torch  ", torch.__version__, "| cuda", torch.version.cuda, "| gpu", torch.cuda.is_available())print("mmcv   ", mmcv.__version__)print("mmpose ", mmpose.__version__)from mmcv.ops import nms                      # fails loudly if the wheel/torch pair is wrongprint("mmcv CUDA ops OK")

## 3. Code + data

In [ ]:
import os, globIN = glob.glob('/kaggle/input/*')[0]          # your dataset folderprint("dataset:", IN, os.listdir(IN))!mkdir -p /kaggle/working/run!cp -r {IN}/PoseAnything_ours /kaggle/working/run/%cd /kaggle/working/run/PoseAnything_ours!python setup.py develop 2>&1 | tail -3!unzip -oq {IN}/goofy_eval.zip -d /kaggle/working/goofy_eval!ls /kaggle/working/goofy_eval && ls /kaggle/working/goofy_eval/images | wc -l

In [ ]:
# verify the eval set before burning GPU timeimport json, numpy as npd = json.load(open('/kaggle/working/goofy_eval/coco_goofy.json'))a = [x for x in d['annotations'] if x['category_id'] == 7]print("goofy anns:", len(a), "| 21 kpts each:",      all(len(x['keypoints']) == 63 for x in a))print("visible/img: min %d med %d max %d" % tuple(    f(np.array([ (np.array(x['keypoints']).reshape(-1,3)[:,2]>0).sum() for x in a ]))    for f in (np.min, np.median, np.max)))

## 4. The actual test`--steps 0` = no adaptation. `--steps 5` = support-guided TTA.The script prints both columns in one run, so the delta is the answer.

In [ ]:
CKPT = '/kaggle/input/CHANGEME/ckpt/selftrain_r4_ep40.pth'   # <-- set this!python test_tta.py   --config configs/cartoon/train_merged_supcon.py   --checkpoint {CKPT}   --coco /kaggle/working/goofy_eval/coco_goofy.json   --img-dir /kaggle/working/goofy_eval/images   --category 7 --shots 5 --trials 3   --steps 5 --lr 1e-4 --mode norm

## 5. If the loss barely moves / PCK dropsRead the per-trial line `loss A -> B`:| symptom | change ||---|---|| `loss` almost flat | `--lr 3e-4` then `1e-3` || `loss` falls but PCK drops | `--steps 2` (overfitting 5 images) || still nothing at `mode=norm` | `--mode norm_head` || high variance across trials | `--trials 5` |

In [ ]:
!python test_tta.py --config configs/cartoon/train_merged_supcon.py --checkpoint {CKPT}   --coco /kaggle/working/goofy_eval/coco_goofy.json --img-dir /kaggle/working/goofy_eval/images   --category 7 --shots 5 --trials 3 --steps 5 --lr 3e-4 --mode norm